In [10]:
"""
Script: 15b_excluir_pacientes_nueva_infeccion.py

Objetivo:
Excluir completamente a los pacientes que presentan al menos
un nuevo cultivo positivo durante el seguimiento, identificado
por la variable `new_pos_culture_flag` en
14_new_positive_cultures_daily.parquet.

Nota metodológica:
Esta exclusión se aplica por simplicidad computacional y para evitar
mezclar la evolución de infecciones distintas. En una versión posterior,
esta lógica podrá sustituirse por censura temporal en el día del evento.
"""

import pandas as pd

# ======================================
# 1. Cargar dataset
# ======================================

df = pd.read_parquet("14_new_positive_cultures_daily.parquet")

print(
    f"Dataset original: {df.shape[0]:,} filas | "
    f"{df['subject_id'].nunique():,} pacientes"
)

# ======================================
# 2. Identificar pacientes con nueva infección
# ======================================

patients_with_new_infection = (
    df.loc[df["new_pos_culture_flag"] == 1, "subject_id"]
      .unique()
)

print(
    f"Pacientes con ≥1 nuevo cultivo positivo: "
    f"{len(patients_with_new_infection):,}"
)

# ======================================
# 3. Excluir pacientes completos
# ======================================

df_filtered = (
    df.loc[~df["subject_id"].isin(patients_with_new_infection)]
      .copy()
)

print(
    f"Dataset tras exclusión: {df_filtered.shape[0]:,} filas | "
    f"{df_filtered['subject_id'].nunique():,} pacientes"
)

# ======================================
# 4. Chequeos de seguridad
# ======================================

assert (
    df_filtered["new_pos_culture_flag"].sum() == 0
), "ERROR: Quedan eventos de nuevo cultivo positivo tras la exclusión"

assert (
    df_filtered["subject_id"].nunique()
    <= df["subject_id"].nunique()
), "ERROR: Inconsistencia en número de pacientes"

print("Chequeos de integridad OK")

# ======================================
# 5. Guardar dataset filtrado
# ======================================

df_filtered.to_parquet("15_dataset_diario_sin_nueva_infeccion.parquet",index=False)

print(
    "Dataset filtrado guardado correctamente.\n"
    "Listo para usar como input del script 16."
)

Dataset original: 1,130,610 filas | 15,938 pacientes
Pacientes con ≥1 nuevo cultivo positivo: 2,659
Dataset tras exclusión: 501,369 filas | 13,279 pacientes
Chequeos de integridad OK
Dataset filtrado guardado correctamente.
Listo para usar como input del script 16.
